# Unlimited-OCR Works: Google Colab Runner
This notebook runs the official **`baidu/Unlimited-OCR`** (3.3B MoE) model on a free Google Colab GPU (T4 / A100).

**GitHub Repository:** [https://github.com/baidu/Unlimited-OCR](https://github.com/baidu/Unlimited-OCR)  
**Hugging Face:** [https://huggingface.co/baidu/Unlimited-OCR](https://huggingface.co/baidu/Unlimited-OCR)

> **Note:** Make sure GPU is enabled: Go to **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU**.

In [ ]:
# 1. Check GPU availability
!nvidia-smi

In [ ]:
# 2. Install required dependencies
!pip install -q torch torchvision transformers pillow einops addict easydict pymupdf psutil accelerate

In [ ]:
# 3. Load the official baidu/Unlimited-OCR model from Hugging Face
import torch
from transformers import AutoModel, AutoTokenizer

model_name = 'baidu/Unlimited-OCR'
print("Loading tokenizer and model weights from Hugging Face...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=torch.bfloat16,
    device_map="auto"
).eval()

print("Unlimited-OCR Model loaded successfully on GPU!")

In [ ]:
# 4. Helper function: Convert PDF pages to images
import os, tempfile, fitz

def pdf_to_images(pdf_path, dpi=200):
    doc = fitz.open(pdf_path)
    tmp_dir = tempfile.mkdtemp(prefix='pdf_ocr_')
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    paths = []
    for i, page in enumerate(doc):
        out = os.path.join(tmp_dir, f'page_{i+1:04d}.png')
        page.get_pixmap(matrix=mat).save(out)
        paths.append(out)
    doc.close()
    return paths

In [ ]:
# 5. Upload your document (PDF or Image)
from google.colab import files
print("Upload your PDF or Image:")
uploaded = files.upload()
uploaded_filename = list(uploaded.keys())[0]
print(f"Selected file: {uploaded_filename}")

In [ ]:
# 6. Run Unlimited-OCR Inference!
output_dir = './ocr_output'
os.makedirs(output_dir, exist_ok=True)

if uploaded_filename.lower().endswith('.pdf'):
    img_files = pdf_to_images(uploaded_filename, dpi=200)
    print(f"Running multi-page parsing on {len(img_files)} pages...")
    result = model.infer_multi(
        tokenizer,
        prompt='<image>Multi page parsing.',
        image_files=img_files,
        output_path=output_dir,
        image_size=1024,
        max_length=32768,
        no_repeat_ngram_size=35,
        ngram_window=1024,
        save_results=True
    )
else:
    print("Running single-image parsing...")
    result = model.infer(
        tokenizer,
        prompt='<image>document parsing.',
        image_file=uploaded_filename,
        output_path=output_dir,
        base_size=1024,
        image_size=640,
        crop_mode=True,
        max_length=32768,
        no_repeat_ngram_size=35,
        ngram_window=128,
        save_results=True
    )

print("\n--- OCR Extraction Completed! ---")
print("Output saved in:", output_dir)

In [ ]:
# 7. Display parsed results
output_files = os.listdir(output_dir)
print("Generated files:", output_files)
for f in output_files:
    if f.endswith('.txt') or f.endswith('.md'):
        with open(os.path.join(output_dir, f), 'r', encoding='utf-8') as fp:
            print(f"\n=== {f} ===")
            print(fp.read()[:2000])